# GenCare AI: Data generatie
---
Auteur:   Eva Rombouts  
Datum:    december 2023  
Script:   Dit script genereert fictieve zorgdata met behulp van de OpenAI API. 

Let op. Het instellen van een OpenAI API key is vereist (https://platform.openai.com/docs/quickstart?context=python)
Het genereren van 24 clientenprofielen met gpt-3.5-turbo kost ongeveer 3 cent. Met gpt-4 krijg je aanzienlijk betere modellen, dan betaal je 70 cent. 
Voor 10 weken rapportages van 24 clienten betaal je 70 cent met 3.5 turbo.

---

## Setup

In [72]:
import pandas as pd
import re
import json 
from openai import OpenAI 
import time

In [123]:
seed = 6
aantal_clienten = 24
num_weeks = 10
model_profielen = 'gpt-4'
model_rapportages = 'gpt-3.5-turbo'
filename_clienten = '../zorgdata/perenboom_profielen.json'
filename_rapportages = '../zorgdata/perenboom_rapportages.json'

## Data generatie functie

Onderstaande functie produceert AI-gegenereerde data.

Allereerst wordt een instantie gemaakt van de OpenAI-client. Hiermee wordt verbinding gemaakt met de OpenAI API.  

Vervolgens genereert chat.completions.create() de tekst.  
Voorbeeld van OpenAI:  
openai.chat.completions.create(  
model="gpt-3.5-turbo",  
  messages=[  
        {"role": "system", "content": "You are a helpful assistant."},  
        {"role": "user", "content": "Who won the world series in 2020?"},  
        {"role": "assistant", "content": "The Los Angeles Dodgers won the World Series in 2020."},  
        {"role": "user", "content": "Where was it played?"}  
    ]  
)

Argumenten:  
- s_role: De system rol bepaalt hoe de 'assistent' zich gedraagt. Je kan instructies meegeven over zijn doel en manier van antwoorden. De system rol wordt slechts één keer gedefinieerd.  
- u_role: De prompt van de 'user' waar de 'assistent' een antwoord (of een completion) op geeft. Eventueel kan dit de start zijn van een dialoog, maar dat gebruiken wij niet. 
- model: Voor mogelijke modellen zie: https://platform.openai.com/docs/models
- seed: spreekt voor zich
- n: Aantal completions die hij teruggeeft

In [112]:
def genereer_zorgdata(s_role, u_role, model='gpt-3.5-turbo', seed=None, n=1):
    try:
        client = OpenAI()
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": s_role},
                {"role": "user", "content": u_role}
            ],
            seed=seed,
            n=n
        )
        return completion
    except Exception as e:
        print(f"Er is een fout opgetreden: {e}")
        return None

## Genereer cliëntprofielen
Definieer inhoud rollen

In [47]:
s_role_profielen = '''
Je specialiseert je in het genereren van fictieve gegevens voor helpend en verzorgend personeel in verpleeghuizen, met een focus op cliënten met een psychogeriatrische aandoening. Jouw expertise omvat het creëren van realistische cliëntscenario's, cliëntendossiers en zorgplannen die specifiek zijn afgestemd op de behoeften van deze doelgroep. Deze gegevens moeten toegankelijk en relevant zijn voor personeel zonder uitgebreide medische kennis, en moeten belangrijke aspecten van de dagelijkse zorg en ondersteuning voor deze cliënten bevatten.
'''

u_role_profielen = '''
Schrijf een profiel van een cliënt die is opgenomen op een psychogeriatrische afdeling van het verpleeghuis. Gebruik onderstaande structuur:

Naam: [Meneer/Mevrouw Voornaam Achternaam]
Type dementie: [kies uit: Alzheimer, gemengde dementie, vasculaire dementie, lewy body dementie, parkinsondementie, FTD, hou rekening met hoe vaak deze vormen van dementie in de populatie voorkomen]
Lichamelijke klachten: [lichamelijke klachten]
Beschrijving cliënt: [een korte beschrijving van karakter en relevante biografische gegevens]
Belangrijkste zorgbehoefte:
- ADL: [Beschrijf ADL hulp]
- cognitie / probleemgedrag: [beschrijf voor de zorg relevante aspecten van cognitie en probleemgedrag. Varieer met de ernst van het probleemgedrag van rustige cliënten, gemiddeld onrustige clienten tot cliënten die fors apathisch, onrustig, angstig, geagiteerd of zelfs agressief kunnen zijn]

Vermijd veelvoorkomende namen als Jansen, de Vries en Fatima.
Vermijd stereotypen in beroepen en achtergronden. Niet elke vrouw was lerares en niet elke man was bankdirecteur of architect.
'''

In [48]:
perenboom = genereer_zorgdata(s_role=s_role_profielen, u_role=u_role_profielen, model=model_profielen, seed=seed, n=aantal_clienten)

In [49]:
# Wijzig formaat naar een dictionary voor opslag als json
perenboom_profielen = {
    "model": perenboom.model,
    "clienten": [{"profiel": choice.message.content} for choice in perenboom.choices],
}

# Sla op als json
with open(filename_clienten, 'w') as file:
    json.dump(perenboom_profielen, file)

In [113]:
df_perenboom_profielen = pd.DataFrame([{"profiel": choice.message.content} for choice in perenboom.choices])

In [120]:
df_perenboom_profielen['client_id'] = df_perenboom_profielen['profiel'].apply(lambda x: x.split('\n')[0])

In [121]:
df_perenboom_profielen.head()

,profiel,client_id
0,Naam: Meneer Hendrik Koster\nType dementie: Va...,Naam: Meneer Hendrik Koster
1,Naam: Meneer Leo Kruisman\nType dementie: vasc...,Naam: Meneer Leo Kruisman
2,Naam: Mevrouw Leonie Berkhof\nType dementie: L...,Naam: Mevrouw Leonie Berkhof
3,Naam: Meneer Julius de Wit\nType dementie: Vas...,Naam: Meneer Julius de Wit
4,Naam: Meneer Henk Breeman\nType dementie: Lewy...,Naam: Meneer Henk Breeman


## Genereer rapportages

In [118]:
# Definieer inhoud rollen
s_role_rapportages = '''
Je specialiseert je nu in het genereren van fictieve gegevens voor helpend en verzorgend personeel in verpleeghuizen, met een focus op cliënten met een psychogeriatrische aandoening. Jouw expertise omvat het creëren van realistische cliëntscenario's, cliëntendossiers en zorgplannen die specifiek zijn afgestemd op de behoeften van deze doelgroep. Deze gegevens moeten toegankelijk en relevant zijn voor personeel zonder uitgebreide medische kennis, en moeten belangrijke aspecten van de dagelijkse zorg en ondersteuning voor deze cliënten bevatten.
'''

u_role_rapportages_instruction = '''
\n
Genereer fictieve zorgrapportages voor deze cliënt voor een periode van zeven dagen. 
De rapportages moeten afwisselend geschreven worden door een helpende (20% kans), een verzorgende (60% kans), of een verpleegkundige (20% kans). Zorg voor variatie in de tijdstippen van de dag waarop de rapportages zijn geschreven, met een focus op overdag, en in mindere mate 's avonds en 's nachts. 
In de inhoud van elke rapportage komt meestal één, soms meer eigenschappen of beperkingen van de cliënt naar voren. Dit kunnen functionele of lichamelijke beperkingen zijn. En soms onrust. Wissel hiermee af, niet elke rapportage gaat over gedrag.

Geef elke rapportage een onrustscore: Een heel getal tussen 0 en 100, wat de mate weergeeft waarin de cliënt onrust vertoont in deze rapportage:
0-20: Geen onrust
21-40: Lichte onrust
41-60: Matige onrust
61-80: Ernstige onrust
81-100: Extreme onrust
Als een rapportage bijvoorbeeld over e

Elke rapportage dient de volgende structuur te hebben: 
---StartRapportage---
Dag: [Dag van de week]
Niveau: [Helpende/Verzorgende/Verpleegkundige]
Onrustscore: [Onrustscore]
Rapportage: [Inhoud van de rapportage]
---EindRapportage---
'''

In [ ]:
# Print een lijstje van de clienten. Daarmee kan de voortgang worden bijgehouden
for ct in perenboom_profielen['clienten']:
    print(ct['profiel'].split('\n')[0])

In [119]:
rapportage_list = []
for index, row in df_perenboom_profielen.iterrows():
    # Voeg de instructietekst toe aan het profiel
    u_role_rapportages = row['profiel'] + u_role_rapportages_instruction

    # Genereer de rapportages
    rapportages = genereer_zorgdata(s_role=s_role_rapportages, u_role=u_role_rapportages, model=model_rapportages, seed=seed, n=num_weeks)

    # Voeg elke rapportage toe aan de lijst
    for rapportage in rapportages.choices:
        rapportage_list.append({
            'client_id': row['client_id'],  # of een andere unieke identificator
            'rapportage': rapportage.message.content,
        })


Naam: Meneer Hendrik Koster
Type dementie: Vasculaire dementie
Lichamelijke klachten: Artritis met name in de handen; blijvende kortademigheid na een oude longontsteking; hypertensie.

Beschrijving cliënt: 
Als voormalige havenarbeider in de haven van Rotterdam, houdt meneer Koster van het buitenleven en is hij gewend aan fysieke arbeid. Nu hij ouder is en lijdt aan vasculaire dementie, is zijn energieniveau echter aanzienlijk verminderd. Hij heeft altijd in Rotterdam gewoond en heeft een sterke liefde voor zijn stad. Hij heeft twee volwassen kinderen, maar zijn vrouw is enkele jaren geleden overleden. Meneer Koster is grotendeels een rustige, zorgzame man maar kan soms ongeduldig en gefrustreerd raken.

Belangrijkste zorgbehoefte:

- ADL: Meneer Koster heeft hulp nodig bij het aankleden vanwege artritis in zijn handen. Hij slaagt er vaak niet in zijn knopen dicht te doen en vindt het moeilijk om zijn sokken aan te trekken. Hij heeft ook hulp nodig bij het wassen en drogen, met name he

In [ ]:
# Maak een lege dictionary aan voor alle cliënten
perenboom_rapportages = {}

for ct in perenboom_profielen['clienten']:
    start = time.time()
    client_id = ct['profiel'].split('\n')[0] # Op de eerste regel staat de naam
    print (client_id) # Print om voortgang bij te houden
    u_role_content = ct['profiel'] + u_role_rapportages_instruction # Voeg het clientprofiel toe aan de instructie
    
    #Genereer de rapportages
    ct_rapportages = genereer_zorgdata(s_role=s_role_rapportages, u_role=u_role_content, model=model_rapportages, seed=seed, n=num_weeks)

    # Maak een lege dictionary voor de rapportages van deze cliënt
    client_rapportages = {}
    # Itereer over elke 'choice' (week) in de response
    for j in range(len(ct_rapportages.choices)):
        # Gebruik de index als weeknummer
        weeknummer = f"Week {j+1}"
        # Sla de inhoud van de response op voor deze week
        client_rapportages[weeknummer] = ct_rapportages.choices[j].message.content
    
    # Voeg het profiel van de cliënt toe aan de dictionary
    client_rapportages['Profiel'] = ct['profiel']
    # Voeg de rapportage dictionary voor deze cliënt toe aan de hoofddictionary
    perenboom_rapportages[client_id] = client_rapportages
    print (f"Verstreken tijd: {round(time.time()-start)} seconden")

In [61]:
# Nu bevat perenboom_rapportages de rapportages voor elke week voor elke cliënt
# Sla op als json
with open(filename_rapportages, 'w') as file:
    json.dump(perenboom_rapportages, file)

In [62]:
perenboom_clienten = {}

for i, (key, value) in enumerate(perenboom_rapportages.items()):
    uniek_id = f"kamer{i+1:02d}"  # Genereert ID's zoals 'kamer01', 'kamer02', etc.
    naam = key.replace('Naam: ', '') # Verwijder prefix
    naam_geslacht = naam.split(' ', 1)
    if 'Meneer' in naam_geslacht[0]:
        geslacht = 'man' 
        naam = naam_geslacht[1]
    elif 'Mevrouw' in naam_geslacht[0]:
        geslacht = 'vrouw' 
        naam = naam_geslacht[1]
    else:
        geslacht = 'onbekend'
    
    perenboom_clienten[uniek_id] = {
        "naam": naam,
        "geslacht": geslacht,
        "profiel": value.get("Profiel", ""),
        "rapportages": {k: v for k, v in value.items() if k.startswith("Week")}
    }


In [66]:
# Maak een DataFrame voor de cliënten
df_clienten = pd.DataFrame.from_dict(perenboom_clienten, orient='index')
df_clienten.reset_index(inplace=True)
df_clienten.rename(columns={'index': 'kamernummer'}, inplace=True)
df_clienten.drop(columns=['rapportages'], inplace=True)

In [67]:
def split_rapportage(rapportage_tekst):
    # Combineer start- en eindpatronen in één patroon
    patroon = r'\s*-+\s*(startrapportage|eindrapportage)\s*-+\s*'
    splitpatroon = r'[\n\s]*§+[\n\s]*'

    # Vervang zowel start- als eindpatronen door §
    rapportage = re.sub(patroon, '§', rapportage_tekst, flags=re.IGNORECASE)

    # Verwijder § aan het begin en het einde (indien aanwezig)
    rapportage = re.sub(r'^' + splitpatroon, '', rapportage)
    rapportage = re.sub(splitpatroon + r'$', '', rapportage)

    # Voer een split uit
    rapportages = re.split(splitpatroon, rapportage)
    return rapportages


In [101]:
def parse_dagrapportage(dagrapportage):
    # Zorg dat de onrustscore altijd op een nieuwe regel begint
    rapportage = re.sub(r'\n*(onrustscore)', '\nOnrustscore', dagrapportage, flags=re.IGNORECASE)
    # Zorg dat de rapportage direct achter de tekst 'Rapportage:' begint (dus zonder newline)
    rapportage = re.sub(r'\n*(rapportage:)[\n\s]*', '\nRapportage: ', rapportage, flags=re.IGNORECASE)

    rapportage_delen = re.split(r'\n', rapportage)

    parsed_data = {
        "dag": None,
        "niveau": None,
        "rapportage": None,
        "onrustscore": None
    }

    for deel in rapportage_delen:
        if deel.startswith('Dag:'):
            parsed_data["dag"] = deel[len('Dag:'):].strip().lower()
        elif deel.startswith('Niveau:'):
            parsed_data["niveau"] = deel[len('Niveau:'):].strip()
        # elif deel.startswith('Onrustscore:'):
        #     parsed_data["onrustscore"] = deel[len('Onrustscore:'):].strip()
        elif deel.startswith('Rapportage:'):
            parsed_data["rapportage"] = deel[len('Rapportage:'):].strip()
        elif deel.startswith('Onrustscore:'):
            # Zet de onrustscore om naar een integer
            score = deel[len('Onrustscore:'):].strip()
            try:
                parsed_data["onrustscore"] = int(score)
            except ValueError:
                # Handel eventuele conversiefouten af
                parsed_data["onrustscore"] = None

    return parsed_data

In [102]:
dagrapportages = []

for uniek_id, client_info in perenboom_clienten.items():
    for week, weekrapportage in client_info['rapportages'].items():
        weekno = week[len('week'):].strip()
        try:
            weekno = int(weekno)
        except ValueError:
            weekno = None
        dagrapportages_per_week = split_rapportage(weekrapportage)
        for dagrapportage in dagrapportages_per_week:
            rpt = parse_dagrapportage(dagrapportage)
            dagrapportages.append({
                "client_id": uniek_id,
                "week": weekno,
                "dag": rpt['dag'],
                "niveau": rpt['niveau'],
                "onrustscore": rpt['onrustscore'],
                "rapportage": rpt['rapportage']
            })

df_rapportages = pd.DataFrame(dagrapportages)


In [109]:
df_clienten.to_csv('../zorgdata/gci_perenboom_clienten.csv', index=False)
df_rapportages.to_csv('../zorgdata/gci_perenboom_rapportages.csv', index=False)